In [1]:
import os
import glob
from tqdm import tqdm
import json
from json.decoder import JSONDecodeError
import copy

In [2]:
from pathlib import Path

In [13]:
import glob
from shutil import copy
files = glob.glob('./models/SPTSv2/output/*/results/*.json')
for file in tqdm(files):
    img_name = file.split('/')[-1]
    img_id = file.split('/')[-3]
    img_name = 'spts_'+img_name
    if not os.path.exists(f'results/totaltext/{img_id}/'):
        os.makedirs(f'results/totaltext/{img_id}/')
    copy(file,f'results/totaltext/{img_id}/{img_name}')

        


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 244.57it/s]


In [3]:
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as F
import torch
from torch.utils.data import DataLoader
class CocoDetection(torchvision.datasets.CocoDetection):
    """

    Args:
        torchvision (_type_): _description_
    """
    def __init__(self, img_folder, ann_file, dataset_name,filter_id):
        super(CocoDetection, self).__init__(img_folder, ann_file)
        self.dataset_name = dataset_name
        self.filter_id = filter_id

    def __getitem__(self, idx):
        img, target = super(CocoDetection, self).__getitem__(idx)
        
        image_id = self.ids[idx]
        image_id = torch.tensor([image_id])
        # print("image_id: ",image_id,"length of target",len(target))
        target = {'image_id': image_id, 'annotations': target}
        target['dataset_name'] = self.dataset_name
        anno = target["annotations"]
        
        target = {}
        anno = [obj for obj in anno if 'iscrowd' not in obj or obj['iscrowd'] == 0]
        boxes = [obj["bbox"] for obj in anno]
        # guard against no boxes via resizing
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        classes = [obj["category_id"] for obj in anno]
        classes = torch.tensor(classes, dtype=torch.int64)
        ids = torch.tensor([obj["id"] for obj in anno], dtype=torch.int64)
        
        target["boxes"] = boxes
        target["labels"] = classes
        target["image_id"] = image_id
        target["filter"] = self.filter_id
        target['dataset_name'] = self.dataset_name
        target['id'] = ids


        # for conversion to coco api
        area = [obj["area"] for obj in anno]
        iscrowd = [obj["iscrowd"] if "iscrowd" in obj else 0 for obj in anno]
        w, h = img.size
        target["orig_size"] =torch.tensor([int(h), int(w)])
        return F.to_tensor(img),target ## Tensors are similar to NumPy’s ndarrays, except that tensors can run on GPUs or other hardware accelerators
    
    
class collate_fn(object):
    def __init__(self):
        pass
    
    def __call__(self, batch):
        batch = list(zip(*batch))
        samples = batch[0]
        targets = batch[1]
        return samples, targets

In [4]:
import glob
import os.path as osp
filter_li = [int(id.split('/')[-1]) for id in glob.glob(r'./data/selected_images/[1-9]*')]
filter_li

[3, 2, 1, 4, 5, 6, 32, 7, 33, 34, 40, 35, 41, 36, 38, 39]

# Azure

In [5]:
!pip install azure-ai-vision-imageanalysis
!pip install pycocotools

In [39]:
import os
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential

# Set the values of your computer vision endpoint and computer vision key
# as environment variables:
try:
    endpoint = "https://vi-ocr.cognitiveservices.azure.com/"
    key = "845e64ae45d845d2ad28ccbf519df527"
except KeyError:
    print("Missing environment variable 'VISION_ENDPOINT' or 'VISION_KEY'")
    print("Set them before running this sample.")
    exit()

# Create an Image Analysis client
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# Get a caption for the image. This will be a synchronously (blocking) call.
# result = client.analyze_from_url(
#     image_url="https://learn.microsoft.com/azure/ai-services/computer-vision/media/quickstarts/presentation.png",
#     visual_features=[VisualFeatures.CAPTION, VisualFeatures.READ],
#     gender_neutral_caption=True,  # Optional (default is False)
# )

# print("Image analysis results:")
# # Print caption results to the console
# # print(" Caption:")
# # if result.caption is not None:
# #     print(f"   '{result.caption.text}', Confidence {result.caption.confidence:.4f}")

# # Print text (OCR) analysis results to the console
# print(" Read:")
# if result.read is not None:
#     for line in result.read.blocks[0].lines:
#         print(f"   Line: '{line.text}', Bounding box {line.bounding_polygon}")
#         for word in line.words:
#             print(f"     Word: '{word.text}', Bounding polygon {word.bounding_polygon}, Confidence {word.confidence:.4f}")

In [40]:
from azure.core.exceptions import HttpResponseError
import time

def acure_detect_text(img_path):
  with open(img_path, "rb") as image_file:
    content = image_file.read()
  try:
    result = client.analyze( image_data= content,
        visual_features=[VisualFeatures.READ],
    )
    # time.sleep(1)
  except HttpResponseError as e:
    print('{} - {}'.format(img_path,e.message))
    return None

  text = []
  boxes = []
  scores = []
  lines = []
  if result.read is not None and len(result.read.blocks)>0:
    for line in result.read.blocks[0].lines:
      # print(f"   Line: '{line.text}', Bounding box {line.bounding_polygon}")
      for word in line.words:
        text.append(word.text)
        boxes.append([dict(point) for point in word.bounding_polygon])
        scores.append(word.confidence)
  # if result.read is not None:
  #     for line in result.read.blocks[0].lines:
  #         print(f"   Line: '{line.text}', Bounding box {line.bounding_polygon}")
  #         for word in line.words:
  #             print(f"     Word: '{word.text}', Bounding polygon {word.bounding_polygon}, Confidence {word.confidence:.4f}")
  else:
      return None
  return {"text":text,
            "boxes":boxes,
            "scores":scores}

In [41]:
from tqdm import tqdm
import os
import json
import numpy as np
dataset_name = 'totaltext'
batch = 1
model = 'azure'
for filter_id in filter_li:
    dataset = CocoDetection(img_folder=osp.join('./data/selected_images/',str(filter_id)),
                        ann_file='./data/selected_images/selected_test.json',
                            dataset_name='totaltext',filter_id = filter_id)

    sampler_val = torch.utils.data.SequentialSampler(dataset) if not dataset is None else None
    val_loader = DataLoader(dataset, batch_size=1,sampler=sampler_val,drop_last=False,collate_fn=collate_fn())
    output_dir = osp.join('./results/totaltext',str(filter_id))
        

    results = []
    err = []

    for imgs, targets in tqdm(val_loader):
        # img = imgs[0]
        # img_arr = (img.permute(1,2,0).squeeze().numpy()*255).astype(np.uint8)
        try:

            # print(preds)
            target = targets[0]
            path = osp.join('./data/selected_images/',str(filter_id),"0000{}.jpg".format(str(target['image_id'].item()).zfill(3)))
            preds = acure_detect_text(path)
        except Exception as e:
            print(target['image_id'].to('cpu').item())
            err.append('id:{},image_id{}'.format(target['id'].to('cpu').item(),
                                                  target['image_id'].to('cpu').item()))
            break
        if preds is None:
            continue
        rec_texts,coords,rec_scores = list(preds.values())
        for text in list(zip(rec_texts,coords,rec_scores)):
            image_id = target['image_id'].to('cpu').item()
            det_scores = 0
            result = {'image_id': image_id,
                      'category_id': 1,
                      'polys': [ list(i.values()) for i in text[1]],
                      'rec_texts': text[0],
                      'det_score':det_scores,
                      'rec_score': text[2],
                      'filter': target['filter'] if 'filter' in target.keys() else 1}
            results.append(result)

    json_path = os.path.join(output_dir, model+'_'+dataset_name+'.json')
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, 'w') as f:
        json.dump(results,f)   
    

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


  5%|█████▊                                                                                                               | 1/20 [00:00<00:02,  7.61it/s]

./data/selected_images/3/0000002.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
./data/selected_images/3/0000018.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
./data/selected_images/3/0000024.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make s

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 64.30it/s]


./data/selected_images/3/0000166.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
./data/selected_images/3/0000181.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
./data/selected_images/3/0000184.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make s

...

./data/selected_images/39/0000002.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
./data/selected_images/39/0000018.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
./data/selected_images/39/0000024.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Mak

 70%|████████████████████████████████████████████████████████████████████████████████▌                                  | 14/20 [00:00<00:00, 133.62it/s]

./data/selected_images/39/0000210.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
./data/selected_images/39/0000245.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
./data/selected_images/39/0000264.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Mak

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 115.92it/s]

./data/selected_images/39/0000299.jpg - (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.
Code: 401
Message: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.


# PPOCR

In [8]:
!python3 -m pip install paddlepaddle-gpu==2.6.0 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
!pip install "paddleocr>=2.0.1" # 推荐使用2.0.1+版本

Looking in links: https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html


In [7]:
from tqdm import tqdm
import os
import json
import numpy as np
from paddleocr import PaddleOCR, draw_ocr

infer = PaddleOCR(use_angle_cls=True, lang="en", ocr_version="PP-OCRv4",use_gpu=True)  # need to run only once to download and load model into memory
dataset_name = 'totaltext'
batch = 1
model = 'ppocr'
for filter_id in filter_li:
    dataset = CocoDetection(img_folder=osp.join('./data/selected_images/',str(filter_id)),
                        ann_file='./data/selected_images/selected_test.json',
                            dataset_name='totaltext',filter_id = filter_id)

    sampler_val = torch.utils.data.SequentialSampler(dataset) if not dataset is None else None
    val_loader = DataLoader(dataset, batch_size=1,sampler=sampler_val,drop_last=False,collate_fn=collate_fn())
    output_dir = osp.join('./results/totaltext',str(filter_id))
    
    results = []
    err = []
    for imgs, targets in tqdm(val_loader):
        img = imgs[0]
        img_arr = (img.permute(1,2,0).squeeze().numpy()*255).astype(np.uint8)
        try:
            det_poly,rec,_ = infer(img_arr,cls=True)
            # print(preds)
            target = targets[0]
        except Exception as e:
            # print(target['image_id'].to('cpu').item())
            err.append('id:{},image_id{}'.format(target['id'].to('cpu').item(),
                                                  target['image_id'].to('cpu').item()))
            break
        image_id = target['image_id'].to('cpu').item()
        for text in list(zip(det_poly,rec)):
            det_scores = 0
            result = {'image_id': image_id,
                            # 'id':target['annotations'][0]['id'].to('cpu'),
                            'category_id': 1,
                            'polys': [i.tolist() for i in text[0]],
                            'rec_texts': text[1][0],
                            # 'score': split_value.mean().item(),
                            # 'value': split_value.numpy().tolist(),
                            'det_score':det_scores,
                            'rec_score': text[1][1],
                            'filter': target['filter'] if 'filter' in target.keys() else 1
                        }
            results.append(result)

    json_path = os.path.join(output_dir, model+'_'+dataset_name+'.json')
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, 'w') as f:
        json.dump(results,f)   
    

[2024/06/06 19:15:28] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=True, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/home/ec2-user/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/home/ec2-user/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, rec

  0%|                                                                                                                             | 0/20 [00:00<?, ?it/s]

[2024/06/06 19:15:31] ppocr DEBUG: dt_boxes num : 0, elapsed : 0.4594838619232178
[2024/06/06 19:15:31] ppocr DEBUG: cls num  : 0, elapsed : 0
[2024/06/06 19:15:31] ppocr DEBUG: rec_res num  : 0, elapsed : 1.430511474609375e-06


  5%|█████▊                                                                                                               | 1/20 [00:00<00:09,  1.98it/s]

[2024/06/06 19:15:31] ppocr DEBUG: dt_boxes num : 0, elapsed : 0.037064552307128906
[2024/06/06 19:15:31] ppocr DEBUG: cls num  : 0, elapsed : 0
[2024/06/06 19:15:31] ppocr DEBUG: rec_res num  : 0, elapsed : 1.430511474609375e-06
[2024/06/06 19:15:31] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.04665112495422363
[2024/06/06 19:15:31] ppocr DEBUG: cls num  : 1, elapsed : 0.029954195022583008
[2024/06/06 19:15:31] ppocr DEBUG: rec_res num  : 1, elapsed : 0.03244781494140625


...

 80%|████████████████████████████████████████████████████████████████████████████████████████████▊                       | 16/20 [00:00<00:00, 25.26it/s]

[2024/06/06 19:15:48] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.03352522850036621
[2024/06/06 19:15:48] ppocr DEBUG: cls num  : 1, elapsed : 0.006364345550537109
[2024/06/06 19:15:48] ppocr DEBUG: rec_res num  : 1, elapsed : 0.008657455444335938
[2024/06/06 19:15:49] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.028322219848632812
[2024/06/06 19:15:49] ppocr DEBUG: cls num  : 1, elapsed : 0.006333112716674805
[2024/06/06 19:15:49] ppocr DEBUG: rec_res num  : 1, elapsed : 0.00862884521484375
[2024/06/06 19:15:49] ppocr DEBUG: dt_boxes num : 3, elapsed : 0.034998416900634766
[2024/06/06 19:15:49] ppocr DEBUG: cls num  : 3, elapsed : 0.006290435791015625
[2024/06/06 19:15:49] ppocr DEBUG: rec_res num  : 3, elapsed : 0.00857090950012207


 95%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████▏     | 19/20 [00:00<00:00, 20.03it/s]

[2024/06/06 19:15:49] ppocr DEBUG: dt_boxes num : 7, elapsed : 0.04590272903442383
[2024/06/06 19:15:49] ppocr DEBUG: cls num  : 7, elapsed : 0.0125274658203125
[2024/06/06 19:15:49] ppocr DEBUG: rec_res num  : 7, elapsed : 0.02062821388244629


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 20.80it/s]


# Google

In [17]:
!pip install google-cloud-vision
!pip install --upgrade google-cloud-vision

In [19]:
from google.cloud import vision

In [14]:
!../google-cloud-sdk/bin/gcloud  init

Welcome! This command will take you through the configuration of gcloud.

Settings from your current configuration [default] are:
core:
  account: kaiagaoqy@gmail.com
  disable_usage_reporting: 'True'
  project: evaocr

Pick configuration to use:
 [1] Re-initialize this configuration [default] with new settings 
 [2] Create a new configuration
Please enter your numeric choice:  ^C


Command killed by keyboard interrupt



In [15]:
!gcloud auth application-default login

/bin/bash: gcloud: command not found


In [17]:
!../google-cloud-sdk/bin/gcloud auth application-default set-quota-project evaocr


Credentials saved to file: [/home/ec2-user/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "evaocr" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [19]:
from google.cloud import vision
import re
def detect_text(path):
    """Detects text in the file."""


    client = vision.ImageAnnotatorClient()

    with open(path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)

    response = client.text_detection(image=image)
    texts = response.text_annotations
    t = []
    boxes = []
    # print("Texts:")
    # print(response)
    # print(response.keys())

    for id, text in enumerate(texts):
        if id ==0:
          # print(id)
          # print(f'\n"{text.description}"')
          t = re.split('\n| ',text.description)

        vertices = [
            f"({vertex.x},{vertex.y})" for vertex in text.bounding_poly.vertices
        ]

        boxes.append(vertices)
        # print("bounds: {}".format(",".join(vertices)))

    if response.error.message:
        raise Exception(
            "{}\nFor more info on error messages, check: "
            "https://cloud.google.com/apis/design/errors".format(response.error.message)
        )
    return {'text':t,'boxes':boxes}


In [38]:


from tqdm import tqdm
import os
import json
import numpy as np
import ast
dataset_name = 'totaltext'
batch = 1
model = 'google'
for filter_id in filter_li:
    dataset = CocoDetection(img_folder=osp.join('./data/selected_images/',str(filter_id)),
                        ann_file='./data/selected_images/selected_test.json',
                            dataset_name='totaltext',filter_id = filter_id)

    sampler_val = torch.utils.data.SequentialSampler(dataset) if not dataset is None else None
    val_loader = DataLoader(dataset, batch_size=1,sampler=sampler_val,drop_last=False,collate_fn=collate_fn())
    output_dir = osp.join('./results/totaltext',str(filter_id))
    
    results = []
    err = []

    for imgs, targets in tqdm(val_loader):
        try:
            target = targets[0]
            path = osp.join('./data/selected_images/',str(filter_id),"0000{}.jpg".format(str(target['image_id'].item()).zfill(3)))
            preds = detect_text(path)
        except Exception as e:
            print(target['image_id'].to('cpu').item())
            err.append('id:{},image_id{}'.format(target['id'].to('cpu').item(),
                                                  target['image_id'].to('cpu').item()))
            break
        if preds is None:
            continue
        rec_texts,coords = list(preds.values())
        for text in list(zip(rec_texts,coords)):
            image_id = target['image_id'].to('cpu').item()
            det_scores = 0
            result = {'image_id': image_id,
                      'category_id': 1,
                      'polys': [ list(ast.literal_eval(text[1][0])) for i in text[1]],
                      'rec_texts': text[0],
                      'det_score':det_scores,
                      'rec_score': 0,
                      'filter': target['filter'] if 'filter' in target.keys() else 1}
            results.append(result)

    json_path = os.path.join(output_dir, model+'_'+dataset_name+'.json')
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, 'w') as f:
        json.dump(results,f)   
    



loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.86it/s]


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.95it/s]


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.08it/s]


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.99it/s]


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.26it/s]


In [37]:
text

('AMBIRA', ['(64,467)', '(581,467)', '(581,589)', '(64,589)'])